# Week 2 exercise — solarinayo

Small **OpenAI Agents SDK** demo: an agent with a **web search** tool (DuckDuckGo) and a **Gradio** UI.

**Setup:** project root `.env` with `OPENAI_API_KEY`, kernel = `Python (agents .venv)`, `uv sync`.

**Screenshot for PR:** run all cells, then run the last cell. When the browser opens, capture the Gradio page with a sample question and the agent reply.

In [ ]:
import json
import os

import gradio as gr
from dotenv import load_dotenv
from duckduckgo_search import DDGS

from agents import Agent, Runner, function_tool, trace

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the repo root .env file")

print("Ready — OPENAI_API_KEY is set.")

In [ ]:
@function_tool
def web_search(query: str) -> str:
    """Search the public web for current facts. Pass a short, specific query."""
    with DDGS() as ddgs:
        hits = list(ddgs.text(query, max_results=5))
    return json.dumps(hits, indent=2)[:8000]

In [ ]:
research_agent = Agent(
    name="ResearchAssistant",
    instructions=(
        "You help with short research questions. "
        "Use web_search when you need fresh or factual information from the web. "
        "Answer clearly in a few paragraphs; cite what you found at a high level."
    ),
    tools=[web_search],
    model="gpt-4o-mini",
)

In [ ]:
async def run_agent(question: str) -> str:
    if not question.strip():
        return "Enter a question first."
    with trace("week2_solarinayo_gradio"):
        result = await Runner.run(research_agent, question.strip())
    return result.final_output


with gr.Blocks(title="Week 2 — Agent + search") as demo:
    gr.Markdown("## Research assistant (OpenAI Agents + DuckDuckGo)")
    q = gr.Textbox(
        label="Your question",
        placeholder="e.g. What is the latest stable Python 3.12 patch release?",
        lines=2,
    )
    go = gr.Button("Run agent", variant="primary")
    out = gr.Markdown(label="Answer")
    go.click(fn=run_agent, inputs=q, outputs=out)
    q.submit(fn=run_agent, inputs=q, outputs=out)

# Opens your browser — use this view for your PR screenshot
demo.launch(inbrowser=True)